In [2]:
import parsers
from common.utils.rclone import copy, list_remote
from subprocess import call
import os
import os
import matlab.engine
import re
import shutil
from common.utils.time import unix_to_timestamps
from common.utils.ingest import storage_format_date
import numpy as np
import dropbox

In [18]:
myRCSParser = parsers.RCSParser(r'/Users/raphaelb/Documents/UW/Research/gridlab/optimal/data/rcs07/rcs/combined_original')

In [3]:
myRCSParser.full_parse()

the MATLAB function has been cancelled
the MATLAB function has been cancelled


In [16]:
os.chdir(r'/Users/raphaelb/Documents/UW/Research/gridlab/optimal/')

In [19]:
#copy(r'/Users/raphaelb/Documents/UW/Research/gridlab/optimal/data-net-subject/source_data/temp/test.txt', 'secret_sauce:/dir4/')

In [55]:
list_of_ucsf_server_sessions = np.asarray(['Session1569834591963', 'Session1588356642957', 'Session1654791220617'])

In [56]:
#Get dates folder names that has been processed and uploaded to wasabi
current_dates = list_remote('rcs07/rcs_v2/')
current_dates = [int(i[0:-2]) for i in current_dates]

#Find the most recent date on wasabi
lastest_date_index = np.argmax(current_dates)
lastest_date = current_dates[lastest_date_index]


#Convert list of sessions (Unix Date -> Timestamp -> Date -> Date as integer)
session_dates = np.asarray([int(unix_to_timestamps(i[7:]).strftime("%Y%m%d")) for i in list_of_ucsf_server_sessions])

#Check for any dates later than the most recent downloaded data
new_sessions_mask = session_dates > lastest_date
new_session_names = list_of_ucsf_server_sessions[new_sessions_mask]

#Download the foldes in the new_session_names folder TODO: when we get ucsf server
print(new_session_names)



['Session1654791220617']


In [58]:
list_remote('rcs07/')

['compiled_v1/\n',
 'curated_datasets/\n',
 'pose_2d/\n',
 'rcs/\n',
 'rcs_v2/\n',
 'video/\n']

In [130]:
from common.utils.rune import get_client, read_fields,get_watch_data,get_bilateral_watch_data,get_watch_data,make_df_from_rune_accessor

In [85]:
myclient = get_client()

In [159]:
time_range = [1653584400,1653604400]
wrist_params = {
'patient_id': 'rcs07',
'left_watch_id': '8QuY9OFb',
'right_watch_id': 'RElEtNme',
'time_range': time_range
}

right_watch_params = {
        'patient_id': 'rcs07',
        'device_id': 'RElEtNme',
        'start_time': time_range[0],
        'end_time': time_range[1]
}
left_watch_params = {
        'patient_id': 'rcs07',
        'device_id': '8QuY9OFb',
        'start_time': time_range[0],
        'end_time': time_range[1]
}

In [126]:
data = get_bilateral_watch_data(myclient, 'dyskinesia', **wrist_params)

In [140]:
all_fields = ['accel','rotation','heart rate','tremor','tremor severity','dyskinesia']

In [169]:

my_accel = get_bilateral_watch_data(myclient, 'accel', **wrist_params)
my_rotation = get_bilateral_watch_data(myclient, 'rotation', **wrist_params)
my_heart_rate = (make_df_from_rune_accessor(myclient.HeartRate(**left_watch_params)),(make_df_from_rune_accessor(myclient.HeartRate(**right_watch_params))))
my_tremor = (make_df_from_rune_accessor(myclient.ProbabilitySymptom(symptom='tremor',**left_watch_params)),(make_df_from_rune_accessor(myclient.ProbabilitySymptom(symptom='tremor',**right_watch_params))))
my_tremor_severity = (make_df_from_rune_accessor(myclient.ProbabilitySymptom(symptom='tremor',severity='*',**left_watch_params)),(make_df_from_rune_accessor(myclient.ProbabilitySymptom(symptom='tremor',severity='*',**right_watch_params))))
my_dyskinesia = (make_df_from_rune_accessor(myclient.ProbabilitySymptom(symptom='dyskinesia',**left_watch_params)),(make_df_from_rune_accessor(myclient.ProbabilitySymptom(symptom='dyskinesia',**right_watch_params))))


(             time  probability
 0    1.653584e+09     0.000000
 1    1.653584e+09     0.000000
 2    1.653585e+09     0.000000
 3    1.653585e+09     0.000000
 4    1.653585e+09     0.000000
 ..            ...          ...
 327  1.653604e+09     0.086957
 328  1.653604e+09     0.000000
 329  1.653604e+09     0.000000
 330  1.653604e+09     0.000000
 331  1.653604e+09     0.000000
 
 [332 rows x 2 columns],
              time  probability
 0    1.653584e+09          0.0
 1    1.653584e+09          0.0
 2    1.653585e+09          0.0
 3    1.653585e+09          0.0
 4    1.653585e+09          0.0
 ..            ...          ...
 329  1.653604e+09          0.0
 330  1.653604e+09          0.0
 331  1.653604e+09          0.0
 332  1.653604e+09          0.0
 333  1.653604e+09          0.0
 
 [334 rows x 2 columns])

In [173]:
print(my_accel)
print(my_rotation)
print(my_heart_rate)
print(my_tremor)
my_dyskinesia[0].to_csv('file_name.csv')

(                     x         y         z
timestamp                                 
1.653584e+09 -0.065415  0.217773 -0.969803
1.653584e+09 -0.063477  0.215926 -0.974457
1.653584e+09 -0.066880  0.217620 -0.974136
1.653584e+09 -0.068207  0.218017 -0.971771
1.653584e+09 -0.067917  0.218871 -0.972214
...                ...       ...       ...
1.653604e+09 -0.169907  0.115188 -0.974228
1.653604e+09 -0.169739  0.123763 -0.992020
1.653604e+09 -0.172989  0.098098 -1.002747
1.653604e+09 -0.178223  0.129928 -0.973083
1.653604e+09 -0.172379  0.150619 -0.973083

[995118 rows x 3 columns],                      x         y         z
timestamp                                 
1.653584e+09 -0.402893 -0.041611 -0.909409
1.653584e+09 -0.413528 -0.068222 -0.910660
1.653584e+09 -0.422241 -0.069962 -0.912308
1.653584e+09 -0.408661 -0.059723 -0.905121
1.653584e+09 -0.408722 -0.088700 -0.899399
...                ...       ...       ...
1.653604e+09  0.360931  0.226089 -0.902054
1.653604e+09  0.362091  0

KeyError: "['x', 'y', 'z'] not in index"

## Dropbox Code

In [23]:
#### DROPBOX_ACCESS_TOKEN = 'sl.BJ_794kQETs-AQ7jWJnc--vK2fCQ6RoPolVzvQ4vZEKdYfmYsr5WinUt3b-BCHlEEKjPuV59oGGaUA2ZPXI3JSyNy2F4MTe5bCVegs8z5WbXes0YZT7BF7H4tzX7K_HJPOrXo_Pc'
def dropbox_connect():
    """Create a connection to Dropbox."""

    try:
        dbx = dropbox.Dropbox(DROPBOX_ACCESS_TOKEN)
    except AuthError as e:
        print('Error connecting to Dropbox with access token: ' + str(e))
    return dbx

def dropbox_list_files():
    """Return a Pandas dataframe of files in a given Dropbox folder path in the Apps directory.
    """

    dbx = dropbox_connect()

    try:
        files = dbx.sharing_list_folders().entries
        files_list = []
        for file in files:
            if isinstance(file, dropbox.files.FileMetadata):
                metadata = {
                    'name': file.name,
                    'path_display': file.path_display,
                    'client_modified': file.client_modified,
                    'server_modified': file.server_modified
                }
                files_list.append(metadata)

        df = pd.DataFrame.from_records(files_list)
        return df.sort_values(by='server_modified', ascending=False)

    except Exception as e:
        print('Error getting list of files from Dropbox: ' + str(e))

dbx = dropbox.Dropbox(DROPBOX_ACCESS_TOKEN)
unscyned_shared_link = dbx.sharing_list_folders().entries[1].preview_url
dbx.files_list_folder('/SummitData/SummitContinuousBilateralStreaming/RCS07L',recursive=False, shared_link=dropbox.files.SharedLink(url=unscyned_shared_link)).entries[456]

FolderMetadata(id='id:cRdrSkVO-GAAAAAAACUSJg', name='Session1651792726456', parent_shared_folder_id='10344256800', path_display=NOT_SET, path_lower=NOT_SET, preview_url=NOT_SET, property_groups=NOT_SET, shared_folder_id=NOT_SET, sharing_info=FolderSharingInfo(no_access=False, parent_shared_folder_id='10344256800', read_only=True, shared_folder_id=NOT_SET, traverse_only=False))

In [26]:
dbx.sharing_get_shared_link_file_to_file('/Users/raphaelb/Documents/UW/Research/gridlab/optimal/data/rcs07/rcs/combined_original',unscyned_shared_link, path='/SummitData/SummitContinuousBilateralStreaming/RCS07L/Session1651792726456/.')



ApiError: ApiError('5ebf395776e84611a7b86191f17072a9', GetSharedLinkFileError('shared_link_not_found', None))